In [ ]:
# [CELL 1] SETUP: STUDENT CONFIGURATION AND DATA LOADING
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Espiritu"            # <-- same surname as main activity
STUDENT_ID = "TUPM-26-12345"      # <-- same student ID as main activity

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

# Paths: data/ stays in the main repo folder; outputs go to the exercise folder
FOLDER_NAME = "Lab7_NumPy_Pandas_Additional_Exercises"
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == FOLDER_NAME else CWD
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / FOLDER_NAME
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw_measurements.shape
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

# Identify features with missing or constant observed values
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_indices = np.flatnonzero(~(all_missing | constant))

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)
print("Data Folder:", DATA_DIR)
print("Output Folder:", OUTPUT_DIR)
print("Shape:", raw_measurements.shape)
print("Features with Missing Values:", missing.any(axis=0).sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

In [ ]:
# [EXERCISE 1] VECTORIZED MEASUREMENT QUALITY ANALYSIS
n_select = 5 + (SEED_DIGIT % 3)
rng1 = np.random.default_rng(RANDOM_SEED)
sel1 = np.sort(rng1.choice(usable_indices, size=n_select, replace=False))
sel1_names = [feature_names[i] for i in sel1]

X1 = raw_measurements[:, sel1]

# Feature-wise statistics (NumPy, ignoring missing values)
means1 = np.nanmean(X1, axis=0)
medians1 = np.nanmedian(X1, axis=0)
sds1 = np.nanstd(X1, axis=0, ddof=1)
missing_pct1 = 100 * np.isnan(X1).mean(axis=0)

# Robust standardization: modified z = 0.6745 * (x - median) / MAD
mad1 = np.nanmedian(np.abs(X1 - medians1), axis=0)
nonzero_mad = mad1 > 0
safe_mad = np.where(nonzero_mad, mad1, 1.0)

# Zero-MAD features: robust z set to 0 (never flagged) to avoid division by zero
robust_z = np.where(nonzero_mad, 0.6745 * (X1 - medians1) / safe_mad, 0.0)

with np.errstate(invalid="ignore"):
    flagged = np.abs(robust_z) > 3.5          # NaN comparisons evaluate False

observed1 = (~np.isnan(X1)).sum(axis=0)
flag_counts = flagged.sum(axis=0)
flag_pct = 100 * flag_counts / observed1      # % of observed values flagged

quality = pd.DataFrame({
    "Feature": sel1_names,
    "Mean": means1,
    "Median": medians1,
    "Sample_SD": sds1,
    "MAD": mad1,
    "Missing_Percentage": missing_pct1,
    "Flagged_Count": flag_counts,
    "Flagged_Percentage": flag_pct,
}).sort_values(["Flagged_Percentage", "Feature"],
               ascending=[False, True], kind="stable").reset_index(drop=True)
quality.insert(0, "Rank", np.arange(1, len(quality) + 1))

# Verify vectorized NumPy results against an independent Pandas calculation
df1 = pd.DataFrame(X1, columns=sel1_names)
pd_mad = (df1 - df1.median()).abs().median()
pd_z = 0.6745 * (df1 - df1.median()) / pd_mad.where(pd_mad > 0)
pd_flag_counts = (pd_z.abs() > 3.5).sum().to_numpy()

vectorized_verified = bool(
    np.allclose(means1, df1.mean().to_numpy())
    and np.allclose(medians1, df1.median().to_numpy())
    and np.allclose(sds1, df1.std().to_numpy())
    and np.allclose(missing_pct1, 100 * df1.isna().mean().to_numpy())
    and np.array_equal(flag_counts, pd_flag_counts)
)

quality.to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)

print("Selected Features:", sel1_names)
print(quality.round(4))
print("--- Assessment Data (Exercise 1) ---")
print("Selected Features:", sel1_names)
print("Highest Missing Percentage:",
      f"{missing_pct1.max():.2f}% ({sel1_names[int(missing_pct1.argmax())]})")
print("Feature with Highest Flagged Percentage:",
      quality.loc[0, "Feature"], f"({quality.loc[0, 'Flagged_Percentage']:.2f}%)")
print("Total Flagged Observations:", int(flag_counts.sum()))
print("Vectorized Results Verified:", "Yes" if vectorized_verified else "No")

In [ ]:
# [EXERCISE 2] OUTCOME COMPARISON WITH GROUPED STATISTICS
# 1. Labeled DataFrame preserving original SECOM record identifiers
full_df = pd.DataFrame(raw_measurements.copy(), columns=feature_names)
full_df.insert(0, "Example_ID", np.arange(len(full_df)))
full_df["Result"] = labels
full_df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

# 2. Reproducible balanced sample
pass_ids = full_df.index[full_df["Outcome"] == "Pass"].to_numpy()
fail_ids = full_df.index[full_df["Outcome"] == "Fail"].to_numpy()
per_group = min(20 + ID_SUM, len(pass_ids), len(fail_ids))

rng2 = np.random.default_rng(RANDOM_SEED)

# 3. Five usable features + median imputation using FULL-dataset medians
sel2 = np.sort(rng2.choice(usable_indices, size=5, replace=False))
sel2_names = [feature_names[i] for i in sel2]

chosen = np.sort(np.concatenate([
    rng2.choice(pass_ids, size=per_group, replace=False),
    rng2.choice(fail_ids, size=per_group, replace=False),
]))

balanced = full_df.loc[chosen, ["Example_ID"] + sel2_names + ["Result", "Outcome"]].copy()
full_medians = full_df[sel2_names].median()
balanced[sel2_names] = balanced[sel2_names].fillna(full_medians)

# 4. Grouped statistics with groupby() and agg()
stats2 = balanced.groupby("Outcome")[sel2_names].agg(["count", "mean", "median", "std"])

# 5-6. Absolute and standardized mean differences (pooled within-group SD)
rows = []
for f in sel2_names:
    n_p, n_f = stats2.loc["Pass", (f, "count")], stats2.loc["Fail", (f, "count")]
    m_p, m_f = stats2.loc["Pass", (f, "mean")], stats2.loc["Fail", (f, "mean")]
    md_p, md_f = stats2.loc["Pass", (f, "median")], stats2.loc["Fail", (f, "median")]
    s_p, s_f = stats2.loc["Pass", (f, "std")], stats2.loc["Fail", (f, "std")]

    pooled_sd = np.sqrt(((n_p - 1) * s_p**2 + (n_f - 1) * s_f**2) / (n_p + n_f - 2))
    abs_diff = abs(m_f - m_p)
    # Zero or invalid denominator handled safely -> 0.0
    std_diff = abs_diff / pooled_sd if np.isfinite(pooled_sd) and pooled_sd > 0 else 0.0

    rows.append({
        "Feature": f, "Pass_Count": int(n_p), "Fail_Count": int(n_f),
        "Pass_Mean": m_p, "Fail_Mean": m_f,
        "Pass_Median": md_p, "Fail_Median": md_f,
        "Pass_SD": s_p, "Fail_SD": s_f,
        "Abs_Mean_Difference": abs_diff, "Pooled_SD": pooled_sd,
        "Standardized_Mean_Difference": std_diff,
    })

comparison = pd.DataFrame(rows).sort_values(
    "Standardized_Mean_Difference", ascending=False, kind="stable"
).reset_index(drop=True)
comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

# Verify Pandas statistics against NumPy
numpy_match = True
for f in sel2_names:
    for grp in ["Pass", "Fail"]:
        v = balanced.loc[balanced["Outcome"] == grp, f].to_numpy()
        numpy_match &= bool(
            np.isclose(v.mean(), stats2.loc[grp, (f, "mean")])
            and np.isclose(np.median(v), stats2.loc[grp, (f, "median")])
            and np.isclose(v.std(ddof=1), stats2.loc[grp, (f, "std")])
            and len(v) == stats2.loc[grp, (f, "count")]
        )

balanced.to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

print(stats2.round(4))
print(comparison.round(4))
print("--- Assessment Data (Exercise 2) ---")
print("Balanced Sample Size:", len(balanced), f"({per_group} per outcome)")
print("Selected Features:", sel2_names)
print("Highest-Ranked Feature:", comparison.loc[0, "Feature"])
print("Absolute Standardized Mean Difference:",
      f"{comparison.loc[0, 'Standardized_Mean_Difference']:.4f}")
print("NumPy-Pandas Statistics Match:", numpy_match)

In [ ]:
# [EXERCISE 3] STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS
pass_mask = labels == -1
fail_mask = labels == 1
pass_idx = np.flatnonzero(pass_mask)
fail_idx = np.flatnonzero(fail_mask)

# 1. Five usable features with >= 10 observed values in EACH outcome group
obs_pass = (~np.isnan(raw_measurements[pass_mask])).sum(axis=0)
obs_fail = (~np.isnan(raw_measurements[fail_mask])).sum(axis=0)
eligible = usable_indices[(obs_pass[usable_indices] >= 10) &
                          (obs_fail[usable_indices] >= 10)]

rng3 = np.random.default_rng(RANDOM_SEED)
sel3 = np.sort(rng3.choice(eligible, size=5, replace=False))
sel3_names = [feature_names[i] for i in sel3]

raw3 = raw_measurements[:, sel3]

# 2. Strategy A: available-case | Strategy B: full-data median imputation
med3 = np.nanmedian(raw3, axis=0)
imp3 = np.where(np.isnan(raw3), med3, raw3)


def mean_diffs(i_fail, i_pass):
    """Fail minus Pass group means for both strategies."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        a = np.nanmean(raw3[i_fail], axis=0) - np.nanmean(raw3[i_pass], axis=0)
    b = imp3[i_fail].mean(axis=0) - imp3[i_pass].mean(axis=0)
    return a, b


# 3. Observed mean differences (Fail - Pass)
obs_a, obs_b = mean_diffs(fail_idx, pass_idx)

# 4. Stratified bootstrap: resample with replacement within each outcome group
B = 200 + 10 * SEED_DIGIT


def run_bootstrap(seed):
    rng = np.random.default_rng(seed)
    boot_a = np.empty((B, len(sel3)))
    boot_b = np.empty((B, len(sel3)))
    for b in range(B):
        bf = rng.choice(fail_idx, size=len(fail_idx), replace=True)
        bp = rng.choice(pass_idx, size=len(pass_idx), replace=True)
        boot_a[b], boot_b[b] = mean_diffs(bf, bp)
    return boot_a, boot_b


boot_a, boot_b = run_bootstrap(RANDOM_SEED)

# 5. Bootstrap mean difference and percentile 95% CI
boot_mean_a = np.nanmean(boot_a, axis=0)
boot_mean_b = np.nanmean(boot_b, axis=0)
ci_a = np.nanpercentile(boot_a, [2.5, 97.5], axis=0)
ci_b = np.nanpercentile(boot_b, [2.5, 97.5], axis=0)

# Standardized differences (pooled SD) for ranking features within each strategy


def pooled_sd(sd_f, n_f, sd_p, n_p):
    return np.sqrt(((n_f - 1) * sd_f**2 + (n_p - 1) * sd_p**2) / (n_f + n_p - 2))


nf_a = (~np.isnan(raw3[fail_idx])).sum(axis=0)
np_a = (~np.isnan(raw3[pass_idx])).sum(axis=0)
psd_a = pooled_sd(np.nanstd(raw3[fail_idx], axis=0, ddof=1), nf_a,
                  np.nanstd(raw3[pass_idx], axis=0, ddof=1), np_a)
psd_b = pooled_sd(imp3[fail_idx].std(axis=0, ddof=1), len(fail_idx),
                  imp3[pass_idx].std(axis=0, ddof=1), len(pass_idx))

std_a = np.where(psd_a > 0, np.abs(obs_a) / np.where(psd_a > 0, psd_a, 1.0), 0.0)
std_b = np.where(psd_b > 0, np.abs(obs_b) / np.where(psd_b > 0, psd_b, 1.0), 0.0)

rank_a = (-std_a).argsort(kind="stable").argsort() + 1
rank_b = (-std_b).argsort(kind="stable").argsort() + 1

# 6. Compare direction, confidence intervals, and rankings across strategies
sign_changed = np.sign(obs_a) != np.sign(obs_b)
rank_changed = rank_a != rank_b
ci_zero_a = (ci_a[0] <= 0) & (ci_a[1] >= 0)
ci_zero_b = (ci_b[0] <= 0) & (ci_b[1] >= 0)
ci_conclusion_changed = ci_zero_a != ci_zero_b

change = np.abs(obs_b - obs_a)
k = int(np.argmax(change))


def direction(x):
    return "Fail > Pass" if x > 0 else ("Fail < Pass" if x < 0 else "Equal")


records = []
for strategy, obs, bm, ci, sd_std, rk, zero in [
    ("Available-Case", obs_a, boot_mean_a, ci_a, std_a, rank_a, ci_zero_a),
    ("Median-Imputed", obs_b, boot_mean_b, ci_b, std_b, rank_b, ci_zero_b),
]:
    for j, f in enumerate(sel3_names):
        records.append({
            "Feature": f, "Strategy": strategy,
            "Observed_Mean_Difference": obs[j],
            "Bootstrap_Mean_Difference": bm[j],
            "CI_Lower_2.5": ci[0, j], "CI_Upper_97.5": ci[1, j],
            "CI_Includes_Zero": bool(zero[j]),
            "Direction": direction(obs[j]),
            "Abs_Standardized_Difference": sd_std[j],
            "Rank": int(rk[j]),
        })

robustness = pd.DataFrame(records)
robustness.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

# 7. Verify reproducibility with the same random seed
boot_a2, boot_b2 = run_bootstrap(RANDOM_SEED)
reproducible = bool(np.array_equal(boot_a, boot_a2, equal_nan=True)
                    and np.array_equal(boot_b, boot_b2, equal_nan=True))

report_lines = [
    "SENSITIVITY AND ROBUSTNESS REPORT",
    f"Student: {LAST_NAME} ({STUDENT_ID})",
    f"Random Seed: {RANDOM_SEED}",
    f"Selected Features: {', '.join(sel3_names)}",
    f"Bootstrap Repetitions: {B} (stratified, with replacement)",
    "Strategies: Available-Case vs Median Imputation (full dataset)",
    "",
    "Per-feature results (Fail - Pass mean difference):",
]
for j, f in enumerate(sel3_names):
    report_lines.append(
        f"- {f}: A={obs_a[j]:.4f} [{ci_a[0, j]:.4f}, {ci_a[1, j]:.4f}] rank {rank_a[j]} | "
        f"B={obs_b[j]:.4f} [{ci_b[0, j]:.4f}, {ci_b[1, j]:.4f}] rank {rank_b[j]}"
    )
report_lines += [
    "",
    f"Largest Change in Mean Difference: {change[k]:.4f} ({sel3_names[k]})",
    f"Features with Changed Difference Sign: {int(sign_changed.sum())}",
    f"Features with Changed Ranking: {int(rank_changed.sum())}",
    f"Features with Changed CI Conclusion (includes 0): {int(ci_conclusion_changed.sum())}",
    f"Reproducibility Verified: {reproducible}",
    "",
    "Interpretation:",
    "- Median imputation fills missing values with a constant, which can shrink "
    "variability and shift group means compared with available-case analysis.",
    "- A CI that includes zero means the group difference is not clearly "
    "separated from sampling variability.",
    "- Group differences are descriptive and do not establish causes of failure.",
    "- Features whose sign, CI conclusion or rank change between strategies "
    "are sensitive to the preparation method.",
]
(OUTPUT_DIR / "sensitivity_report.txt").write_text("\n".join(report_lines) + "\n",
                                                   encoding="utf-8")

print(robustness.round(4))
print("--- Assessment Data (Exercise 3) ---")
print("Selected Features:", sel3_names)
print("Bootstrap Repetitions:", B)
print("Largest Change in Mean Difference:", f"{change[k]:.4f}", f"({sel3_names[k]})")
print("Features with Changed Difference Sign:", int(sign_changed.sum()),
      [sel3_names[i] for i in np.flatnonzero(sign_changed)])
print("Features with Changed Ranking:", int(rank_changed.sum()),
      [sel3_names[i] for i in np.flatnonzero(rank_changed)])
print("Reproducibility Verified:", "Yes" if reproducible else "No")

for p in ["measurement_quality.csv", "balanced_sample.csv", "feature_comparison.csv",
          "robustness_comparison.csv", "sensitivity_report.txt"]:
    print(p, "Created:", (OUTPUT_DIR / p).is_file())